In [47]:
import pandas as pd

df = pd.read_csv("/content/creditcard.csv")

df.head(5)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


# **Identify Outliers using Quartiles**

In [48]:

def get_median(sorted_df, n):
  if n % 2 != 0:
      # print("Odd case")
      #odd case ma directly yeuta median hunxa
      median = sorted_df['Amount'][(n)//2] # used floor division because index should be whole numbers
  else:
      # print("Even case")
      #even case ma duita ko mean lai median maninxa
      median = (sorted_df['Amount'][(n//2)-1] + sorted_df['Amount'][(n//2)])/2
  return median

total_rows = len(df)
print(f"Total rows = {total_rows}")

# paila data lai sort garnu paryo for median
sorted_df = df.sort_values(by=['Amount']).reset_index(drop=True)

lower_half = sorted_df.iloc[:(total_rows // 2)].reset_index(drop=True)

upper_half = sorted_df.iloc[(total_rows // 2):].reset_index(drop=True)

q1 = get_median(lower_half, len(lower_half))
print(f"Q1 ={q1}")

q3 = get_median(upper_half, len(upper_half))
print(f"Q3 ={q3}")

Total rows = 284807
Q1 =5.6
Q3 =77.16499999999999


In [49]:
iqr = q3 - q1
print(f"IQR ={iqr}")

IQR =71.565


In [50]:
upper_limit = q3 + 1.5*iqr
lower_limit = q1 - 1.5*iqr
print(f"Upper Bound ={upper_limit}")
print(f"Lower Bound ={lower_limit}")

outliers = sorted_df[(sorted_df['Amount'] > upper_limit) | (sorted_df['Amount'] < lower_limit)]
# print(f"Outlier ={outliers.values}")
print(f"Total outlier ={len(outliers)}")

Upper Bound =184.5125
Lower Bound =-101.7475
Total outlier =31904


# **Identify Outliers using Standard Deviation**

In [51]:
mean = df['Amount'].sum()/len(df)
print(f"Mean ={mean}")

Mean =88.34961925093133


In [52]:
sqr = 0
for x in df['Amount']:
  sqr += (x-mean)**2 # sum of (x-mean)^2
variance = sqr/(total_rows) # variance calculation
print(f"Variance ={variance}")

Variance =62559.84938857328


In [53]:
sd = variance**(1/2)
print(f"Standard Deviation ={sd}")

Standard Deviation =250.11967013526402


In [54]:
#3 Standard Deviations
upper_limit_std3 = mean + 3*sd
lower_limit_std3 = mean - 3*sd

outliers_std = df[(df['Amount'] < lower_limit_std3) | (df['Amount'] > upper_limit_std3)]
# print(f"Outlier ={outliers_std.values}")
print(f"Number of Outliers (Std Dev): {len(outliers_std)}")
#about 99.7% of data haru chai 3rd standard deviation ma parxa ra beyond that are flagged as outlier

Number of Outliers (Std Dev): 4076


# **Skewness and Kurtosis**

In [55]:
data = df['Amount']

# Central moments
mu1 = (data - mean).sum()/total_rows
mu2 = ((data - mean)**2).sum()/total_rows
mu3 = ((data - mean)**3).sum()/total_rows
mu4 = ((data - mean)**4).sum()/total_rows

print("1st central moment (μ1):", mu1)
print("2nd central moment (μ2 - Variance):", mu2)
print("3rd central moment (μ3):", mu3)
print("4th central moment (μ4):", mu4)

1st central moment (μ1): 5.9268984487409184e-15
2nd central moment (μ2 - Variance): 62559.849388558934
3rd central moment (μ3): 265656676.55592647
4th central moment (μ4): 3319151515527.4185


In [56]:
if mu3 == 0:
  print("Symmetric")
elif mu3 > 0:
  print("Right Skewed")
else:
  print("Left Skewed")

Right Skewed


In [57]:
# Kurtosis
beta2 = mu4 / (mu2**2)

print("Kurtosis (β2):", beta2)

Kurtosis (β2): 848.0777883188754


In [58]:
# Kurtosis type
if beta2 > 3:
    print("Distribution: Leptokurtic")
elif beta2 < 3:
    print("Distribution: Platykurtic")
else:
    print("Distribution: Mesokurtic")

Distribution: Leptokurtic


The transaction data is heavily right-skewed and leptokurtic, meaning most charges are small but a few are extremely large. Because of this "heavy tail," the IQR method is more effective, catching 31,904 outliers compared to only 4,076 found by the Standard Deviation method. Essentially, the extreme values are so large they "stretch" the standard deviation, making it less sensitive to outliers than the quartile approach.